In [1]:
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.nn.functional import scaled_dot_product_attention
import math
import time
import os
from torch.cuda.amp import GradScaler
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist
from transformers import GPT2Tokenizer
from torch.utils.data import DataLoader

class LlamaRMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x): # return x / torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight
        norm = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return self.weight * x / norm
    
class LlamaRotaryEmbedding(nn.Module):
    def __init__(self, dim, base=10000.0):
        super().__init__()
        self.dim = dim
        self.base = base
        # Create inv_freq for half the dimension
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)

    def forward(self, seq_len, device):
        t = torch.arange(seq_len, device=device, dtype=torch.float32)
        sinusoid_inp = torch.einsum('i,j->ij', t, self.inv_freq)  # Outer product
        sin = sinusoid_inp.sin()
        cos = sinusoid_inp.cos()
        return sin, cos

    def apply_rotary_emb(self, x, emb):
        sin, cos = emb
        
        # Reshape sin and cos to match input dimensions
        # Add batch and head dimensions, and ensure the last dimension matches
        sin = sin.view(1, sin.shape[0], 1, sin.shape[1])  # [1, seq_len, 1, dim//2]
        cos = cos.view(1, cos.shape[0], 1, cos.shape[1])  # [1, seq_len, 1, dim//2]
        
        # Split input into half along last dimension
        x_half1, x_half2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        
        # Apply rotary embeddings
        x_out1 = x_half1 * cos - x_half2 * sin
        x_out2 = x_half2 * cos + x_half1 * sin
        
        # Concatenate back along last dimension
        return torch.cat([x_out1, x_out2], dim=-1)
        
class LlamaSdpaAttention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.rotary_emb = LlamaRotaryEmbedding(self.head_dim)

    def forward(self, x, attention_mask=None):
        batch_size, seq_len, _ = x.shape

        # Project queries, keys and values
        q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        k = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        v = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)

        # Apply rotary enbeddings
        rotary_emb = self.rotary_emb(seq_len, x.device)
        q = self.rotary_emb.apply_rotary_emb(q, rotary_emb)
        k = self.rotary_emb.apply_rotary_emb(k, rotary_emb)

        # Reshape for attention
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # Scaled dot product attention
        attn_output = scaled_dot_product_attention(
            q, k, v,
            attn_mask=attention_mask,
            dropout_p=0.0,
            is_causal=True
        )

        # Reshape and project output
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_size)
        return self.o_proj(attn_output)

class LlamaMLP(nn.Module):
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
        self.act_fn = nn.SiLU()

    def forward(self, x):
        return self.down_proj(self.act_fn(self.gate_proj(x) + self.up_proj(x)))
    
class LlamaDecoderLayer(nn.Module):
    def __init__(self, hidden_size, num_heads, intermediate_size):
        super().__init__()
        self.attn = LlamaSdpaAttention(hidden_size, num_heads)
        self.input_layernorm = LlamaRMSNorm(hidden_size)
        self.post_attention_layernorm = LlamaRMSNorm(hidden_size)
        self.mlp = LlamaMLP(hidden_size, intermediate_size)
    
    def forward(self, x, attention_mask=None):
        # Self-attention
        residual = x
        x = self.attn(self.input_layernorm(x), attention_mask)
        x = x + residual

        # Feedforward
        residual = x
        x = self.mlp(self.post_attention_layernorm(x))
        x = x + residual

        return x
    
class LlamaModel(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_hidden_layers, num_heads, intermediate_size):
        super().__init__()
        self.embed_tokens = nn.Embedding(vocab_size, hidden_size)
        self.layers = nn.ModuleList([
            LlamaDecoderLayer(hidden_size, num_heads, intermediate_size)
            for _ in range(num_hidden_layers)
        ])
        self.norm = LlamaRMSNorm(hidden_size)
    
    def forward(self, input_ids, attention_mask=None):

        x = self.embed_tokens(input_ids)

        for layer in self.layers:
            x = layer(x, attention_mask)

        return self.norm(x)
    
class LlamaForCausalLM(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_hidden_layers, num_heads, intermediate_size, max_seq_len):
        super().__init__()
        self.model = LlamaModel(vocab_size, hidden_size, num_hidden_layers, num_heads, intermediate_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)
        self.lm_head.weight = self.model.embed_tokens.weight # Weight Sharing

    def forward(self, input_ids, attention_mask=None):
        x = self.model(input_ids, attention_mask)
        logits = self.lm_head(x)
        return logits
    
    def resize_token_embeddings(self, new_num_tokens):
        """"""
        # Resize the token embeddings
        old_embeddings = self.model.embed_tokens
        new_emmbeddings = nn.Embedding(new_num_tokens, old_embeddings.embedding_dim)

        # Copy the exisiting embeddings
        num_tokens_to_copy = min(old_embeddings.num_embeddings, new_num_tokens)
        new_emmbeddings.weight.data[:num_tokens_to_copy, :] = old_embeddings.weight.data[:num_tokens_to_copy, :]

        # Replace the old embeddings
        self.model.embed_tokens = new_emmbeddings

        # Update the lm_head to match the new embedding size
        old_lm_head = self.lm_head
        self.lm_head = nn.Linear(old_lm_head.in_features, new_num_tokens, bias=False)

        # Copy the exisiting weights
        num_tokens_to_copy = min(old_lm_head.out_features, new_num_tokens)
        self.lm_head.weight.data[:num_tokens_to_copy, :] = old_lm_head.weight.data[:num_tokens_to_copy, :]

        # Maintain weight sharing
        self.lm_head.weight = self.model.embed_tokens.weight

        return self.model.embed_tokens
    
# First load the tokenizer and set pad token
tokenizer = GPT2Tokenizer.from_pretrained("HuggingFaceTB/cosmo2-tokenizer")
tokenizer.pad_token = tokenizer.eos_token
model_vocab_size = len(tokenizer)

# Now create model with the correct vocab size from the start
vocab_size = model_vocab_size
hidden_size = 576
num_hidden_layers = 30
num_heads = 9
intermediate_size = 1536
max_seq_len = 2048

# Instantiate the model with correct vocab size
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LlamaForCausalLM(vocab_size, hidden_size, num_hidden_layers, num_heads, intermediate_size, max_seq_len)

# Print model size
model_size = sum(p.numel() for p in model.parameters())
print(f"Model Size: {model_size / 1e6:.2f}M parameters.")

# Load Cosmopedia Dataset
def load_cosmopedia_dataset(batch_size, seq_length):
    try:
        dataset = load_dataset(
            "HuggingFaceTB/smollm-corpus",
            name="cosmopedia-v2",
            split="train",
            streaming=True
        )

        def encode(example):
            tokens = tokenizer(
                example["text"],
                truncation=True,
                max_length = seq_length,
                return_tensors="pt",
                padding="max_length",
            )
            # Use clone().detach() as recommended
            input_ids = tokens['input_ids'].squeeze(0).clone().detach()
            input_ids = torch.clamp(input_ids, max=vocab_size - 1)
            return {
                'input_ids': input_ids,
                'labels': input_ids.clone().detach()
            }
        
        dataset = dataset.map(encode, remove_columns=['text'], batched=False)
        dataset = dataset.with_format("torch")
        dataloader = DataLoader(dataset, batch_size=batch_size)
        return dataloader
    except Exception as e:
        print(f"Error loading dataset: {str(e)}")
        dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train", streaming=True)
        return load_wikitext_dataset(dataset, batch_size, seq_length)
    
def load_wikitext_dataset(dataset, batch_size, seq_length):
    def encode(example):
        tokens = tokenizer(
            example['text'],
            truncation=True,
            max_length = seq_length,
            return_tensors="pt",
            padding="max_length",
        )
        # Use clone().detach() as recommended
        input_ids = tokens['input_ids'].squeeze(0).clone().detach()
        input_ids = torch.clamp(input_ids, max=vocab_size - 1)
        return {
            'input_ids': input_ids,
            'labels': input_ids.clone().detach()
        }
    
    dataset = dataset.map(encode, remove_columns=['text'], batched=False)
    dataset = dataset.with_format("torch")
    dataloader = DataLoader(dataset, batch_size=batch_size, num_workers=10)
    return dataloader

print("Loading Dataset ...")
train_loader = load_cosmopedia_dataset(batch_size=8, seq_length=1024)

print("\nSample texts from dataset:")
sample_texts = []
sample_count = 3
for i, batch in enumerate(train_loader):
    if i >= sample_count:
        break

    sample_idx = torch.randint(0, batch['input_ids'].size(0), (1,)).item()
    decoded_text = tokenizer.decode(batch['input_ids'][sample_idx], skip_special_tokens=True)
    # Get first 10 words as prompt
    prompt = ' '.join(decoded_text.split()[:10])
    sample_texts.append({
        'full_text': decoded_text[:200] + " ... ",
        'prompt': prompt
    })
    print(f"\nSample {i+1}:")
    print(f"Full text: {sample_texts[-1]['full_text']}")
    print(f"Prompt: {sample_texts[-1]['prompt']}")

print("\nStarting training...")

torch.set_float32_matmul_precision('high')

# Training setup
def setup_training(model, batch_size=8):
    """"Hi"""
    # Enable cudnn benchmarking and deterministic mode
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

    # Move the model to GPU
    device = torch.device("cuda")
    model = model.to(device)

    # Enable gradient scaler for mixed precision training
    scaler = GradScaler()

    # Optimize with larger batch size and lr
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-4,
        betas=(0.9, 0.95),
        eps=1e-8,
        weight_decay=0.1
    )

    # Learning rate scheduler with longer warmup
    lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=3e-4,
        epochs=1,
        steps_per_epoch=10000,
        pct_start=0.1,
        anneal_strategy='cos'
    )

    return device, optimizer, lr_scheduler, scaler

def save_checkpoint(model, optimizer, lr_scheduler, step, loss, save_dir="checkpoints"):

    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    checkpoint = {
        'step': step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': lr_scheduler.state_dict(),
        'loss': loss
    }

    path = os.path.join(save_dir, f'model_step_{step}.pt')
    torch.save(checkpoint, path)
    print(f"Checkpoint saved: {path}")

def load_checkpoint(model, optimizer, lr_scheduler, chkpt_path, device):
    if os.path.exists(chkpt_path):
        checkpoint = torch.load(chkpt_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        lr_scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        step = checkpoint['step']
        loss = checkpoint['loss']
        print(f"Checkpoint loaded: {chkpt_path}")
        print(f"Resumed from step {step}, loss: {loss}")
        return step
    else:
        raise FileNotFoundError(f"Checkpoint file not found: {chkpt_path}")

# Training metrics tracking
best_loss = float('inf')
running_loss = 0.0
log_interval = 1

# Setup training optimizations
device, optimizer, lr_scheduler, scaler = setup_training(model)

# Use the same batch size
train_loader = load_cosmopedia_dataset(batch_size=8, seq_length=1024)

# Add gradient accumulation
gradient_accumulation_steps = 8

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

# Training loop
print("Starting training...")
model.train()
num_epochs = 1  # Only one epoch since we are limiting by steps
max_steps = 5050  # Limit training to 5000 steps
running_loss = 0.0
optimizer.zero_grad()


# Checkpoint path and resume logic
resume_training = True  # Set to True to resume training
chkpt_path = "checkpoints/model_step_5000.pt"  # Adjust path as needed

if resume_training and os.path.exists(chkpt_path):
    step = load_checkpoint(model, optimizer, lr_scheduler, chkpt_path, device)
else:
    step = 0  # Start from scratch

# Add accumulation timing
accum_start_time = time.time()

print("Starting training...")
model.train()

for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1} of {num_epochs}")
    for batch_idx, batch in enumerate(train_loader):
        if step >= max_steps:
            print("Reached max steps. Ending training.")
            break

        if batch_idx % gradient_accumulation_steps == 0:
            accum_start_time = time.time()

        t0 = time.time()

        input_ids, labels = batch['input_ids'], batch['labels']
        input_ids, labels = input_ids.to(device), labels.to(device)

        # AMP
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(input_ids)
            shift_logits = outputs[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()  # Use shifted labels for causal LM
            loss = criterion(shift_logits.view(-1, vocab_size), shift_labels.view(-1))
            loss = loss / gradient_accumulation_steps

        loss.backward()

        if (batch_idx + 1) % gradient_accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            torch.cuda.synchronize()
            step += 1

            # Update metrics
            running_loss += loss.item() * gradient_accumulation_steps

            # Timing and logging for complete accumulated steps
            accum_end_time = time.time()
            total_step_time = (accum_end_time - accum_start_time) * 1000
            batch_tokens = input_ids.size(0) * input_ids.size(1)
            total_tokens = batch_tokens * gradient_accumulation_steps
            tokens_per_sec = total_tokens / (accum_end_time - accum_start_time)

            if step % log_interval == 0:
                avg_loss = running_loss / log_interval
                print(f"Step {step} | Loss: {avg_loss:.4f} | LR: {lr_scheduler.get_last_lr()[0]:.6f} | "
                      f"Total Step Time: {total_step_time:.2f}ms | Tokens/sec: {tokens_per_sec:.2f} "
                      f"(accumulated over {gradient_accumulation_steps} batches)")
                running_loss = 0.0

            # Save checkpoint every 500 steps
            if step > 0 and step % 500 == 0:
                save_checkpoint(model, optimizer, lr_scheduler, step, loss.item())

            # Generate sample text for validation every 100 steps
            if step > 0 and step % 100 == 0:
                model.eval()
                with torch.no_grad():
                    for i, sample in enumerate(sample_texts):
                        print(f"\nGeneration {i+1} with prompt: {sample['prompt']}")
                        inputs = tokenizer(sample['prompt'], return_tensors="pt", padding=True, truncation=True).to(device)
                        outputs = model(inputs["input_ids"])
                        prediction = torch.argmax(outputs, dim=-1)
                        decoded = tokenizer.decode(prediction[0], skip_special_tokens=True)
                        print(f"Generated: {decoded}\n")
                model.train()

    if step >= max_steps:
        break

# Save final checkpoint
save_checkpoint(model, optimizer, lr_scheduler, step, loss.item())

Model Size: 147.79M parameters.
Loading Dataset ...


Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]


Sample texts from dataset:


/home/jovyan/env/lib/python3.10/site-packages/datasets/formatting/torch_formatter.py:87: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(value, **{**default_dtype, **self.torch_tensor_kwargs})


...

Starting training...


/tmp/ipykernel_244387/1891832789.py:348: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(chkpt_path, map_location=device)


Checkpoint loaded: checkpoints/model_step_5000.pt
Resumed from step 5000, loss: 0.45450544357299805
Starting training...
Epoch 1 of 1


/home/jovyan/env/lib/python3.10/site-packages/datasets/formatting/torch_formatter.py:87: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(value, **{**default_dtype, **self.torch_tensor_kwargs})


Step 5001 | Loss: 3.8698 | LR: 0.000176 | Total Step Time: 1914.50ms | Tokens/sec: 34231.34 (accumulated over 8 batches)
Step 5002 | Loss: 3.4856 | LR: 0.000176 | Total Step Time: 938.41ms | Tokens/sec: 69836.92 (accumulated over 8 batches)
Step 5003 | Loss: 3.6295 | LR: 0.000176 | Total Step Time: 917.84ms | Tokens/sec: 71402.20 (accumulated over 8 batches)
Step 5004 | Loss: 4.1896 | LR: 0.000176 | Total Step Time: 930.03ms | Tokens/sec: 70466.86 (accumulated over 8 batches)
Step 5005 | Loss: 3.6039 | LR: 0.000176 | Total Step Time: 899.01ms | Tokens/sec: 72897.60 (accumulated over 8 batches)
Step 5006 | Loss: 4.2050 | LR: 0.000176 | Total Step Time: 922.69ms | Tokens/sec: 71027.13 (accumulated over 8 batches)
Step 5007 | Loss: 3.8855 | LR: 0.000176 | Total Step Time: 908.08ms | Tokens/sec: 72170.20 (accumulated over 8 batches)
Step 5008 | Loss: 3.8074 | LR: 0.000176 | Total Step Time: 900.69ms | Tokens/sec: 72761.62 (accumulated over 8 batches)
Step 5009 | Loss: 3.7416 | LR: 0.000176